In [1]:
import pyspark.sql.functions as f
from pyspark.sql.types import *
from pyspark.ml.feature import VectorAssembler, OneHotEncoder, StringIndexer, Bucketizer
from pyspark.ml import Pipeline
from pyspark.ml.regression import GeneralizedLinearRegression, RandomForestRegressor, DecisionTreeRegressor
from pyspark.ml.evaluation import RegressionEvaluator

In [2]:
from pathlib import Path
path = Path.cwd()
if not Path(path, "data").exists(): path = path.parent

In [3]:
import sys
from pathlib import Path
root = Path.cwd().parent 
sys.path.append(str(root))

from minio_utils import MinioSparkClient
import os
spark = MinioSparkClient(endpoint=os.getenv("MINIO_ENDPOINT", "").replace("http://", "").replace("https://", ""),
                        access_key=os.getenv("MINIO_ACCESS_KEY"),
                        secret_key=os.getenv("MINIO_SECRET_KEY"),
                        memory = 16,
                        heapsize = 8,
                        num_part = 2000,
                        bucket_name="pd2",
                        base_dir="cityenjoyer")
                        
spark.connect()

## Carga y división en train, test

In [4]:
df_tip = spark.read_parquet("tip_model_data/*")
df_tip.show(5)

+--------+------------------+------------+------------+------------+----------------+----------------+------------+------------+---+----+-----+----------+
|VendorID|trip_distance_cbrt|payment_type|PUAskingRent|DOAskingRent|PUlandmark_count|DOlandmark_count|PUis_airport|DOis_airport|dow|hour|month|tip_amount|
+--------+------------------+------------+------------+------------+----------------+----------------+------------+------------+---+----+-----+----------+
|       0|14.665174320150205|           1|      6300.0|      3995.0|              24|              24|           0|           0|  4|  16|   10|       407|
|       0|15.710955610759562|           1|      3995.0|      5088.5|              24|              18|           0|           0|  4|  16|   10|       100|
|       0|11.313305651645821|           1|      3902.5|      3902.5|               8|               8|           0|           0|  4|  16|   10|       100|
|       0|14.094597464129784|           1|      3902.5|      4595.0|  

In [6]:
df_tip.groupBy("VendorID").count().collect()

[Row(VendorID=3, count=64285883),
 Row(VendorID=1, count=534380),
 Row(VendorID=2, count=166473591),
 Row(VendorID=0, count=43995121)]

In [6]:
df_tip.select(f.min("tip_amount"), f.max("tip_amount")).collect()

[Row(min(tip_amount)=0, max(tip_amount)=10000)]

In [9]:
df_tip = df_tip.filter(f.col("tip_amount") <= 50.0).filter("VendorID == 0").sample(.25, 42)

In [10]:
train_df, test_df = df_tip.randomSplit([.7,.3], seed=42)

train_df.cache()
test_df.cache()

print(f"Datos de entrenamiento: {train_df.count()} registros")
print(f"Datos de prueba: {test_df.count()} registros")

Datos de entrenamiento: 758023 registros
Datos de prueba: 324924 registros


In [ ]:
train_df.select(['tip_amount']).describe().show()

+-------+------------------+
|summary|        tip_amount|
+-------+------------------+
|  count|         192697621|
|   mean|132.42502610865134|
| stddev| 316.3332411644987|
|    min|                 0|
|    max|             10000|
+-------+------------------+



In [6]:
test_df.select(['tip_amount']).describe().show()

+-------+------------------+
|summary|        tip_amount|
+-------+------------------+
|  count|          82591354|
|   mean|132.42229351270836|
| stddev| 316.3248080998809|
|    min|                 0|
|    max|             10000|
+-------+------------------+



## Crear flujo de procesamiento

In [31]:
categorical_cols = ["payment_type", "PUis_airport", "DOis_airport", "dow", "hour", "month"] # "VendorID", 
numeric_cols = ["trip_distance_cbrt", "PUAskingRent", "DOAskingRent", "PUlandmark_count", "DOlandmark_count"]
columnas_landmark = ["PUlandmark_count", "DOlandmark_count"]
objective = "tip_amount"

columnas_indexadas = [f"{c}_idx" for c in categorical_cols]
indexer = StringIndexer(
    inputCols=categorical_cols,
    outputCols=columnas_indexadas,
    handleInvalid="keep"
)

columnas_one_hot = [f"{c}_ohe" for c in categorical_cols]
encoder = OneHotEncoder(
    inputCols=columnas_indexadas,
    outputCols=columnas_one_hot,
    dropLast=True
)

columnas_bucket = ["PUlandmark_bucket", "DOlandmark_bucket"]
cortes = [-float("inf"), 1.0, 3.0, 10.0, float("inf")]
bucketizer = Bucketizer(
    splitsArray=[cortes, cortes], 
    inputCols=columnas_landmark, 
    outputCols=columnas_bucket
)

feature_cols = columnas_one_hot  + numeric_cols # + columnas_bucket
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)
assembler

VectorAssembler_ff0417f92fa1

## Modelo GLM
Probamos con un GLM con el parámetro gamma ya que la variable objetivo veíamos que era bastante asimétrica hacia la derecha.

In [36]:
glm = GeneralizedLinearRegression(
    featuresCol="features", 
    labelCol=objective, 
    family="tweedie",
    variancePower=1.5, # 1.5 es el estándar para datos con ceros y asimetría
    linkPower=1      # Equivale al link="log" en Tweedie
)

glm_pipeline = Pipeline(stages=[
    indexer, encoder, bucketizer, assembler, glm
])
print("Pipeline GLM creado con", len(glm_pipeline.getStages()), "etapas")

Pipeline GLM creado con 5 etapas


In [37]:
glm_model = glm_pipeline.fit(train_df)
glm_predictions = glm_model.transform(test_df)

# El fallo es posible que esté relacionado con linkPower o que haya que escalar la variable objetivo
print("Primeras predicciones:")
glm_predictions.select(objective, "features", "prediction").show(10)

Primeras predicciones:
+----------+--------------------+-------------------+
|tip_amount|            features|         prediction|
+----------+--------------------+-------------------+
|         0|(56,[0,5,7,13,17,...|0.20998332349007565|
|         0|(56,[0,5,7,14,16,...|0.20295794866352934|
|         0|(56,[0,5,7,12,17,...| 0.2037148894824437|
|         0|(56,[0,5,7,12,23,...| 0.1989274243388351|
|         0|(56,[0,5,7,10,16,...|0.19348153428848366|
|         0|(56,[0,5,7,12,16,...|0.20264368427790394|
|         0|(56,[0,5,7,13,30,...| 0.2376796146124589|
|         0|(56,[0,5,7,14,23,...| 0.1949118450545612|
|         0|(56,[0,5,7,15,16,...|0.20226451434941672|
|         0|(56,[0,5,7,13,19,...|0.20769592858878227|
+----------+--------------------+-------------------+
only showing top 10 rows


In [38]:
lr_model = glm_model.stages[-1]

# Obtener coeficientes (es un DenseVector)
coefficients = lr_model.coefficients
intercept = lr_model.intercept

# Convertir a array de NumPy para trabajar más fácil
import numpy as np
coef_array = coefficients.toArray()

print(f"Intercepto: {intercept}")
print(f"Coeficientes: {coef_array}")

Intercepto: 0.16638044465413884
Coeficientes: [ 3.02409636e-02 -2.33332822e-02  7.43326603e-01 -1.33397312e-02
 -5.30701823e-02 -6.15830191e-02  6.15830191e-02 -1.52845144e-03
  1.52845144e-03  8.11014572e-03 -1.13937723e-02  1.55543124e-02
 -7.05358895e-03 -4.41233392e-03 -4.66972359e-03  2.69269099e-03
 -1.27382534e-02  5.98629806e-03  1.56524204e-02 -5.92423569e-03
  1.82471544e-02  1.54746760e-03  9.56597483e-04 -9.90749997e-03
  6.38588551e-03  1.09278693e-02  1.20638775e-02  3.76916592e-04
 -6.01489259e-05 -3.75107171e-03  3.66548301e-02  3.25469637e-03
 -6.10747756e-03  9.53138983e-03  1.20271847e-02 -1.18848005e-02
 -1.28183902e-02 -8.86829402e-03 -2.81401504e-02 -2.91101178e-02
  1.45152454e-02 -2.22023414e-03 -7.50416250e-03 -2.04988558e-02
 -2.85970445e-03 -1.04728009e-02 -1.89591610e-02  5.60965361e-02
  2.74357764e-02  3.03423144e-02  1.04733678e-01 -1.26702133e-02
  2.76370960e-05  1.58097265e-05  1.04490843e-04 -2.23122754e-04]


In [39]:
rmse_evaluator = RegressionEvaluator(
    labelCol=objective,
    predictionCol="prediction",
    metricName="rmse"
)

mae_evaluator = RegressionEvaluator(
    labelCol=objective,
    predictionCol="prediction",
    metricName="mae"
)

r2_evaluator = RegressionEvaluator(
    labelCol=objective,
    predictionCol="prediction",
    metricName="r2"
)

glm_rmse = rmse_evaluator.evaluate(glm_predictions)
glm_mae = mae_evaluator.evaluate(glm_predictions)
glm_r2 = r2_evaluator.evaluate(glm_predictions)

print('Métricas test')
print(f"RMSE (Root Mean Squared Error): {glm_rmse:.2f} $")
print(f"MAE (Mean Absolute Error):      {glm_mae:.2f} $")
print(f"R2 (R-squared):                 {glm_r2:.4f}")

Métricas test
RMSE (Root Mean Squared Error): 2.04 $
MAE (Mean Absolute Error):      0.32 $
R2 (R-squared):                 0.0181


In [ ]:
glm_model.write().overwrite().save(str(path / "data" / "glm_model_v1"))

## Decision Tree

In [15]:
feature_cols_tree = columnas_indexadas + columnas_bucket + numeric_cols
assembler_tree = VectorAssembler(
    inputCols=feature_cols_tree,
    outputCol="features"
)
assembler_tree

VectorAssembler_8d07e0b712c5

In [16]:
dt = DecisionTreeRegressor(
    featuresCol="features", 
    labelCol=objective, 
    maxDepth=10
)

dt_pipeline = Pipeline(stages=[
    indexer, bucketizer, assembler_tree, dt
])

print("Entrenando un único Árbol de Decisión...")
dt_model = dt_pipeline.fit(train_df)
dt_predictions = dt_model.transform(test_df)

dt_predictions.select(objective, "features", "prediction").show(10)

Entrenando un único Árbol de Decisión...
+----------+--------------------+--------------------+
|tip_amount|            features|          prediction|
+----------+--------------------+--------------------+
|         0|[0.0,0.0,0.0,0.0,...|0.006984642646190195|
|         0|(12,[4,6,8,10,11]...|0.006984642646190195|
|         0|(12,[4,6,7,10,11]...|0.006984642646190195|
|         0|[0.0,0.0,0.0,0.0,...|0.006984642646190195|
|         0|(12,[4,6,8,10,11]...|0.006984642646190195|
|         0|[0.0,0.0,0.0,0.0,...|0.006984642646190195|
|         0|(12,[4,5,6,10,11]...| 0.04541719578695749|
|         0|(12,[4,6,7,8,10,1...|0.006984642646190195|
|         0|[0.0,0.0,0.0,0.0,...|0.003980891719745223|
|         0|[0.0,0.0,0.0,0.0,...|0.006984642646190195|
+----------+--------------------+--------------------+
only showing top 10 rows


In [17]:
rmse_evaluator = RegressionEvaluator(
    labelCol=objective,
    predictionCol="prediction",
    metricName="rmse"
)

mae_evaluator = RegressionEvaluator(
    labelCol=objective,
    predictionCol="prediction",
    metricName="mae"
)

r2_evaluator = RegressionEvaluator(
    labelCol=objective,
    predictionCol="prediction",
    metricName="r2"
)

dt_rmse = rmse_evaluator.evaluate(dt_predictions)
dt_mae = mae_evaluator.evaluate(dt_predictions)
dt_r2 = r2_evaluator.evaluate(dt_predictions)

print('Métricas test')
print(f"RMSE (Root Mean Squared Error): {dt_rmse:.2f} $")
print(f"MAE (Mean Absolute Error):      {dt_mae:.2f} $")
print(f"R2 (R-squared):                 {dt_r2:.4f}")

Métricas test
RMSE (Root Mean Squared Error): 2.08 $
MAE (Mean Absolute Error):      0.22 $
R2 (R-squared):                 0.0159


## Random Forest

In [33]:
feature_cols_tree = columnas_indexadas + numeric_cols # + columnas_bucket
assembler_tree = VectorAssembler(
    inputCols=feature_cols_tree,
    outputCol="features"
)
assembler_tree

VectorAssembler_e5e5dfecf9ab

In [40]:
rf = RandomForestRegressor(
    featuresCol="features", 
    labelCol=objective, 
    numTrees=150,
    maxDepth=10,
    weightCol="importancia",
    seed=42
)

rf_pipeline = Pipeline(stages=[
    indexer, bucketizer, assembler_tree, rf
])

print("Pipeline de Random Forest creado con", len(rf_pipeline.getStages()), "etapas")

# 3. Entrenamos y predecimos
rf_model = rf_pipeline.fit(train_df)
rf_predictions = rf_model.transform(test_df)

print("Primeras predicciones con Random Forest:")
rf_predictions.select(objective, "features", "prediction").show(10)

Pipeline de Random Forest creado con 4 etapas
Primeras predicciones con Random Forest:
+----------+--------------------+--------------------+
|tip_amount|            features|          prediction|
+----------+--------------------+--------------------+
|         0|[0.0,0.0,0.0,4.0,...|5.562590979409383E-4|
|         0|[0.0,0.0,0.0,5.0,...| 0.00483575841300889|
|         0|[0.0,0.0,0.0,3.0,...|6.715465162864898E-4|
|         0|[0.0,0.0,0.0,3.0,...|4.145270140926773E-4|
|         0|(11,[3,5,7,8,9],[...|5.992796142270947E-4|
|         0|(11,[3,5,7,8,9],[...|6.866211018182623E-4|
|         0|[0.0,0.0,0.0,4.0,...|0.006216933802799...|
|         0|[0.0,0.0,0.0,5.0,...|3.776734531786063E-4|
|         0|[0.0,0.0,0.0,6.0,...|5.503672523092955E-4|
|         0|(11,[3,4,5,7,8],[...| 0.00150405079770698|
+----------+--------------------+--------------------+
only showing top 10 rows


In [41]:
rmse_evaluator = RegressionEvaluator(
    labelCol=objective,
    predictionCol="prediction",
    metricName="rmse"
)

mae_evaluator = RegressionEvaluator(
    labelCol=objective,
    predictionCol="prediction",
    metricName="mae"
)

r2_evaluator = RegressionEvaluator(
    labelCol=objective,
    predictionCol="prediction",
    metricName="r2"
)

rf_rmse = rmse_evaluator.evaluate(rf_predictions)
rf_mae = mae_evaluator.evaluate(rf_predictions)
rf_r2 = r2_evaluator.evaluate(rf_predictions)

print('Métricas test')
print(f"RMSE (Root Mean Squared Error): {rf_rmse:.2f} $")
print(f"MAE (Mean Absolute Error):      {rf_mae:.2f} $")
print(f"R2 (R-squared):                 {rf_r2:.4f}")

Métricas test
RMSE (Root Mean Squared Error): 2.02 $
MAE (Mean Absolute Error):      0.26 $
R2 (R-squared):                 0.0351


Este es el mejor resultado hasta ahora (15/03/2026 17:27)

In [ ]:
rf_model.write().overwrite().save(str(path / "data" / "rf_model_v1"))